# Knee PD / PDFS: intensity histograms and offline ESPIRiT support

Two questions before the guided-recon sweep:

1. **Scale.** What per-contrast `scale_fac` puts the *foreground* 90th–95th percentile near 1?
   (Part A — pooled histograms, per-slice spread, and what the normalized images look like.)
2. **Support.** Does the offline ESPIRiT support (`thresh_eig 0.95`) cut into the anatomy, and
   is 0.9 enough? (Part B — eigenvalue map λ, support outlines at each threshold, maps and the
   SENSE target they produce.)

Everything is read at **raw scale** (`scale_fac = 1`). The maths is imported from
`scripts/contrast_histograms.py` and `scripts/check_espirit_support.py`; this notebook is the
interactive front end (the scripts are the headless versions). Part B needs a GPU for the
640-line grid.

In [ ]:
%matplotlib inline
import json
import os
import pathlib
import sys

import h5py
import numpy as np
import torch
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd()
if not (ROOT / "physics").is_dir():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from datasets.fastmri.paired import PairedKneeDataset
from operators.fourier import ifftc
from physics.smaps import espirit
from scripts.contrast_histograms import foreground, rss, sample_slices
from scripts.check_espirit_support import slice_metrics

print("root", ROOT)

In [ ]:
# ===================== WHAT TO LOOK AT -- TUNE =====================
CONFIG            = "config/knee/guided/tgvn_R8_g1.json"   # split CSV, k-space roots, slice range
SPLIT             = "train"
N_VOLUMES         = 12          # random volumes for the histograms
SLICES_PER_VOLUME = 6           # evenly spaced over the config's training slice range
PERCENTILES       = [90, 95, 99]
SEED              = 0

# offline maps on disk (the PDFS SENSE target). OLD = thresh_eig 0.95, NEW = what the configs use.
OLD_MAPS = "../datasets/fastmri_preprocessed/knee_coil_combined/pdfs_espirit_acs32"
NEW_MAPS = None                 # None -> the config's smap_root (.../pdfs_espirit_acs32_eig0.9)
TARGET_MAPS = "new"             # which maps give the pdfs_sense histogram: "new" | "old"

# ESPIRiT, as torch/espirit_knee_pairs.sbatch runs it
ACS, KERNEL, ROWSPACE = 32, 8, 0.05
THRESHOLDS = [0.95, 0.9, 0.85, 0.8, 0.7]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

cfg = json.load(open(CONFIG))
d = cfg["data"][SPLIT]
NEW_MAPS = NEW_MAPS or d.get("smap_root")
CUR_SCALE = float(d.get("scale_fac", 1.0))
ds = PairedKneeDataset(d["split_csv"], d["kspace_roots"], scale_fac=1.0, target="rss")
rng = np.random.default_rng(SEED)
pairs = [ds.pairs[i] for i in sorted(rng.choice(len(ds.pairs), min(N_VOLUMES, len(ds.pairs)),
                                                 replace=False))]
maps_dir = NEW_MAPS if TARGET_MAPS == "new" else OLD_MAPS
print(f"{len(pairs)}/{len(ds.pairs)} {SPLIT} volumes; configs' scale_fac {CUR_SCALE:g}; "
      f"target maps {maps_dir} (exists: {os.path.isdir(maps_dir or '')})")

## Part A — intensity histograms

Per sampled slice: `pdfs_sense` (|offline SENSE image|, the target; only if the maps exist),
`pdfs_rss`, `pd_rss` (the guide contrast). Foreground = Otsu on each contrast's own RSS.

In [ ]:
def load_slices(pair, sls):
    pd_path, fs_path, n, _ = pair
    with h5py.File(fs_path, "r") as f:
        k_fs = torch.from_numpy(f["kspace"][sls])
    with h5py.File(pd_path, "r") as f:
        k_pd = torch.from_numpy(f["kspace"][sls])
    sense = None
    mp = os.path.join(maps_dir, os.path.basename(fs_path)) if maps_dir else None
    if mp and os.path.exists(mp):
        with h5py.File(mp, "r") as f:
            sense = torch.from_numpy(f["image"][sls]).abs().float()
    return k_fs, k_pd, sense

data = []          # one dict per slice: images (raw), foregrounds, volume / slice ids
for vi, pair in enumerate(pairs):
    sls = sample_slices(pair[2], int(d.get("start_slice", 0)), d.get("end_slice"),
                        SLICES_PER_VOLUME)
    k_fs, k_pd, sense = load_slices(pair, sls)
    for j, sl in enumerate(sls):
        r_fs, r_pd = rss(k_fs[j]), rss(k_pd[j])
        e = dict(vol=vi, name=os.path.basename(pair[1]), slice=sl,
                 pdfs_rss=r_fs, pd_rss=r_pd, fg_pdfs=foreground(r_fs), fg_pd=foreground(r_pd))
        if sense is not None:
            e["pdfs_sense"] = sense[j]
        data.append(e)
CONTRASTS = [c for c in ("pdfs_sense", "pdfs_rss", "pd_rss") if c in data[0]]
FG = {"pdfs_sense": "fg_pdfs", "pdfs_rss": "fg_pdfs", "pd_rss": "fg_pd"}
print(f"{len(data)} slices; contrasts {CONTRASTS}")

In [ ]:
pooled = {c: (np.concatenate([e[c][e[FG[c]]].numpy() for e in data]),
              np.concatenate([e[c].flatten().numpy() for e in data])) for c in CONTRASTS}

print(f"{'contrast':<12}" + "".join(f"{'fg p' + format(q, 'g'):>12}" for q in PERCENTILES)
      + "   scale_fac putting fg p -> 1: " + "  ".join(f"p{q:g}" for q in PERCENTILES))
for c in CONTRASTS:
    pv = [np.percentile(pooled[c][0], q) for q in PERCENTILES]
    print(f"{c:<12}" + "".join(f"{x:12.3e}" for x in pv) + "   "
          + "  ".join(f"{1 / x:9.4g}" for x in pv))
if "pdfs_sense" in pooled:
    for q in PERCENTILES:
        r = np.percentile(pooled["pd_rss"][0], q) / np.percentile(pooled["pdfs_sense"][0], q)
        print(f"PD / PDFS fg p{q:g}: {r:.2f}x")

In [ ]:
LOG_Y = True
fig, ax = plt.subplots(1, len(CONTRASTS), figsize=(6 * len(CONTRASTS), 4), squeeze=False)
for a, c in zip(ax[0], CONTRASTS):
    fg, al = pooled[c]
    p95 = np.percentile(fg, 95)
    bins = np.linspace(0, np.percentile(fg, 99.9) * 1.3 / p95, 200)   # x in units of fg p95
    a.hist(al / p95, bins=bins, histtype="step", color="0.5", density=True, label="whole image")
    a.hist(fg / p95, bins=bins, alpha=0.5, density=True, label="foreground")
    for j, q in enumerate(PERCENTILES):
        a.axvline(np.percentile(fg, q) / p95, ls="--", lw=1.2, color=f"C{j + 2}",
                  label=f"fg p{q:g}")
    a.set_title(f"{c}  (x axis: |image| / fg p95 = {p95:.3e})")
    if LOG_Y:
        a.set_yscale("log")
    a.legend(fontsize=8)
plt.tight_layout()

Per-slice foreground p95, one line per volume: how much a single fixed factor per contrast
leaves unnormalized (edge slices vs centre, volume to volume).

In [ ]:
fig, ax = plt.subplots(1, len(CONTRASTS), figsize=(6 * len(CONTRASTS), 3.5), squeeze=False)
for a, c in zip(ax[0], CONTRASTS):
    ref = np.percentile(pooled[c][0], 95)
    for vi in range(len(pairs)):
        pts = [(e["slice"], np.percentile(e[c][e[FG[c]]].numpy(), 95) / ref)
               for e in data if e["vol"] == vi and e[FG[c]].any()]
        if pts:
            a.plot(*zip(*pts), "-o", ms=3, lw=1, alpha=0.7)
    a.axhline(1, color="k", lw=0.8)
    a.set_title(f"{c}: per-slice fg p95 / pooled fg p95")
    a.set_xlabel("slice")
plt.tight_layout()

What the images look like after dividing by the pooled foreground percentile `NORM_Q`, shown on
[0, 1] — anything brighter saturates white. Pick the percentile at which only small bright
structures (fat, vessels) clip.

In [ ]:
NORM_Q   = 95
SHOW     = 6            # slices to show (spread over the sample)
CONTRAST = CONTRASTS[0]

ref = np.percentile(pooled[CONTRAST][0], NORM_Q)
idx = np.linspace(0, len(data) - 1, SHOW).round().astype(int)
fig, ax = plt.subplots(2, SHOW, figsize=(2.6 * SHOW, 8))
for j, i in enumerate(idx):
    e = data[i]
    x = e[CONTRAST].numpy() / ref
    ax[0, j].imshow(x, cmap="gray", vmin=0, vmax=1)
    ax[0, j].set_title(f"{e['name'][:14]} sl {e['slice']}\n{100 * (x[e[FG[CONTRAST]].numpy()] > 1).mean():.1f}% fg > 1",
                       fontsize=8)
    ax[1, j].imshow(x > 1, cmap="Reds")
    ax[1, j].contour(e[FG[CONTRAST]].numpy(), levels=[0.5], colors="b", linewidths=0.5)
    ax[1, j].set_title("clipped (red), fg (blue)", fontsize=8)
for a in ax.flat:
    a.set_xticks([]); a.set_yticks([])
plt.suptitle(f"{CONTRAST} / pooled fg p{NORM_Q}  (scale_fac = {1 / ref:.4g})")
plt.tight_layout()

## Part B — offline ESPIRiT support

One slice: ESPIRiT runs once (`return_lam=True`), and every threshold is applied to the same
eigenvalue map λ. `EDGE` slices (near the ends of the volume) are where the support tends to clip.

In [ ]:
VOL   = 0               # index into `pairs`
SLICE = "mid"           # "mid", "edge" (a quarter in from the start), or an int
COIL_SHOW = 6

pd_path, fs_path, n, _ = pairs[VOL]
sl = n // 2 if SLICE == "mid" else (n // 4 if SLICE == "edge" else int(SLICE))
with h5py.File(fs_path, "r") as f:
    k = torch.from_numpy(f["kspace"][sl:sl + 1]).to(DEVICE, torch.complex64)
with torch.no_grad():
    sm, lam = espirit(k, acs_size=(ACS, ACS), kernel_size=KERNEL, thresh_rowspace=ROWSPACE,
                      thresh_eig=min(THRESHOLDS), return_lam=True)
coils = ifftc(k)[0].cpu()
r = coils.abs().pow(2).sum(0).sqrt()
lam = lam[0].float().cpu()
Q = sm[0].cpu()               # maps at the LOWEST threshold; any higher one is Q * (lam > t)
m, fg = slice_metrics(r, lam, THRESHOLDS)
print(f"{os.path.basename(fs_path)} slice {sl}/{n}")
print(f"{'thresh':>7} {'cut':>8} {'cut_e':>9} {'over':>7} {'sup':>7}")
for t in THRESHOLDS:
    print(f"{t:7.2f} {m[t]['cut']:8.4f} {m[t]['cut_e']:9.5f} {m[t]['over']:7.3f} {m[t]['sup']:7.3f}")

In [ ]:
cmap = plt.get_cmap("viridis")
fig, ax = plt.subplots(1, 3, figsize=(17, 8))
ax[0].imshow(r.sqrt(), cmap="gray")
ax[0].contour(fg.numpy(), levels=[0.5], colors="r", linewidths=0.8, linestyles="dotted")
for j, t in enumerate(THRESHOLDS):
    ax[0].contour((lam > t).numpy(), levels=[0.5],
                  colors=[cmap(j / max(len(THRESHOLDS) - 1, 1))], linewidths=1)
ax[0].set_title("sqrt RSS; red dots = Otsu fg; support at " +
                ", ".join(f"{t:g}" for t in THRESHOLDS) + " (dark -> light)", fontsize=9)
im = ax[1].imshow(lam, cmap="magma", vmin=0, vmax=1)
ax[1].set_title("top eigenvalue lam")
plt.colorbar(im, ax=ax[1], fraction=0.04)
ax[2].imshow(r.sqrt() * (lam <= THRESHOLDS[0]), cmap="gray")
ax[2].set_title(f"sqrt RSS OUTSIDE the {THRESHOLDS[0]:g} support (what it zeroes)")
for a in ax:
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout()

The SENSE target `x = Σ_c conj(s_c) c_c` each threshold produces, next to RSS. Inside the support
|x| ≈ RSS (unit-RSS maps); outside it x = 0 — the bottom row shows RSS − |x|, i.e. exactly what
each threshold removes from the ground truth.

In [ ]:
fig, ax = plt.subplots(2, len(THRESHOLDS) + 1, figsize=(3.4 * (len(THRESHOLDS) + 1), 8))
vmax = float(np.percentile(r[fg].numpy(), 99))
ax[0, 0].imshow(r, cmap="gray", vmin=0, vmax=vmax); ax[0, 0].set_title("RSS")
ax[1, 0].axis("off")
for j, t in enumerate(THRESHOLDS, start=1):
    s_t = Q * (lam > t)
    x = (s_t.conj() * coils).sum(0).abs()
    ax[0, j].imshow(x, cmap="gray", vmin=0, vmax=vmax)
    ax[0, j].set_title(f"|SENSE| @ {t:g}")
    ax[1, j].imshow(r - x, cmap="magma", vmin=0, vmax=0.3 * vmax)
    ax[1, j].set_title(f"RSS - |SENSE| (x3)\nlost {m[t]['cut_e']:.2%} of fg signal", fontsize=9)
for a in ax.flat:
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout()

In [ ]:
# |maps| of the strongest coils at the current and the proposed threshold
T_SHOW = [0.95, 0.9]
energy = coils.abs().pow(2).sum(dim=(-2, -1))
top = energy.argsort(descending=True)[:COIL_SHOW]
fig, ax = plt.subplots(len(T_SHOW), COIL_SHOW, figsize=(2.6 * COIL_SHOW, 4.3 * len(T_SHOW)),
                       squeeze=False)
for i, t in enumerate(T_SHOW):
    for j, c in enumerate(top.tolist()):
        ax[i, j].imshow((Q[c] * (lam > t)).abs(), cmap="viridis", vmin=0, vmax=1)
        ax[i, j].set_title(f"|s_{c}| @ {t:g}", fontsize=9)
        ax[i, j].set_xticks([]); ax[i, j].set_yticks([])
plt.tight_layout()

Sanity check against the maps on disk: the 0.95 directory's support should be λ > 0.95 here
(same ACS / kernel / row-space), and the new directory's λ > 0.9 once it is written.

In [ ]:
for label, root, t in (("old", OLD_MAPS, 0.95), ("new", NEW_MAPS, 0.9)):
    p = os.path.join(root or "", os.path.basename(fs_path))
    if not os.path.exists(p):
        print(f"{label}: {p} not on disk")
        continue
    with h5py.File(p, "r") as f:
        disk = torch.from_numpy(f["smaps"][sl]).abs().sum(0) > 0
    here = lam > t
    print(f"{label} ({t:g}): disk support {disk.float().mean():.3f} of FOV, lam>{t:g} "
          f"{here.float().mean():.3f}, disagree on {(disk ^ here).float().mean():.4%} of pixels")

### Through the whole volume

ESPIRiT on every `STRIDE`-th slice of the chosen volume: signal lost per threshold against slice
index. One ESPIRiT call per slice (GPU: seconds each).

In [ ]:
STRIDE = 4
rows = []
with h5py.File(fs_path, "r") as f:
    for s in range(0, n, STRIDE):
        ks = torch.from_numpy(f["kspace"][s:s + 1]).to(DEVICE, torch.complex64)
        with torch.no_grad():
            _, lam_s = espirit(ks, acs_size=(ACS, ACS), kernel_size=KERNEL,
                               thresh_rowspace=ROWSPACE, thresh_eig=min(THRESHOLDS),
                               return_lam=True)
        r_s = ifftc(ks)[0].abs().pow(2).sum(0).sqrt().cpu()
        rows.append((s, slice_metrics(r_s, lam_s[0].float().cpu(), THRESHOLDS)[0]))

fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
for j, t in enumerate(THRESHOLDS):
    col = cmap(j / max(len(THRESHOLDS) - 1, 1))
    ax[0].plot([s for s, _ in rows], [mm[t]["cut_e"] for _, mm in rows], "-o", ms=3,
               color=col, label=f"{t:g}")
    ax[1].plot([s for s, _ in rows], [mm[t]["over"] for _, mm in rows], "-o", ms=3, color=col)
ax[0].set_title("fg signal outside the support (cut_e)"); ax[0].set_xlabel("slice")
ax[0].legend(title="thresh_eig", fontsize=8)
ax[1].set_title("support beyond the fg / fg area (over)"); ax[1].set_xlabel("slice")
plt.tight_layout()